# OASIS INFOBYTE - DATA ANALYTICS INTERNSHIP
## Task 3 - Customer Segmentation Analysis
### Intern: Dineo Ndlovu
### Track: Data Analytics
### Dataset: SuperStore Sales Dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv('../SuperStoreOrders_Cleaned.csv', encoding='latin1')
df.head()

,Ã¯Â»Â¿order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,...,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year
0,AG-2011-2040,2011-01-01,2011-01-06,Standard Class,Toby Braunhardt,Consumer,Constantine,Algeria,Africa,Africa,...,Office Supplies,Storage,"Tenex Lockers, Blue",408,2,0.0,106.140,35.46,Medium,2011
1,IN-2011-47883,2011-01-01,2011-01-08,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,Office Supplies,Supplies,"Acme Trimmer, High Speed",120,3,0.1,36.036,9.72,Medium,2011
2,HU-2011-1220,2011-01-01,2011-01-05,Second Class,Annie Thurman,Consumer,Budapest,Hungary,EMEA,Emea,...,Office Supplies,Storage,"Tenex Box, Single Width",66,4,0.0,29.640,8.17,High,2011
3,IT-2011-3647632,2011-01-01,2011-01-05,Second Class,Eugene Moren,Home Office,Stockholm,Sweden,EU,North,...,Office Supplies,Paper,"Enermax Note Cards, Premium",45,3,0.5,-26.055,4.82,High,2011
4,IN-2011-47883,2011-01-01,2011-01-08,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,Furniture,Furnishings,"Eldon Light Bulb, Duo Pack",114,5,0.1,37.770,4.70,Medium,2011


In [3]:
df['order_date'] = pd.to_datetime(df['order_date'])
snapshot_date = df['order_date'].max() + pd.DateOffset(days=1)

rfm = df.groupby('customer_name').agg({
    'order_date': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'count',
    'sales': 'sum'
}).reset_index()

rfm.columns = ['customer_name', 'Recency', 'Frequency', 'Monetary']
print(rfm.head())

KeyError: "Label(s) ['order_id'] do not exist"

In [ ]:
print("=== RFM STATISTICS ===")
print(rfm[['Recency', 'Frequency', 'Monetary']].describe())

In [ ]:
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])
print("Data normalised successfully")

In [ ]:
inertia = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(rfm_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(8,5))
plt.plot(range(1, 11), inertia, marker='o', color='blue')
plt.title('Elbow Method - Optimal Number of Clusters')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.tight_layout()
plt.show()

In [ ]:
km = KMeans(n_clusters=4, random_state=42)
rfm['Cluster'] = km.fit_predict(rfm_scaled)
print("Clusters assigned successfully")
print(rfm['Cluster'].value_counts())

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(data=rfm, x='Recency', y='Monetary', hue='Cluster', palette='Set1', s=100)
plt.title('Customer Segments - Recency vs Monetary')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(data=rfm, x='Frequency', y='Monetary', hue='Cluster', palette='Set1', s=100)
plt.title('Customer Segments - Frequency vs Monetary')
plt.tight_layout()
plt.show()

In [ ]:
cluster_profile = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean().round(2)
print("=== CLUSTER PROFILES ===")
print(cluster_profile)

In [ ]:
plt.figure(figsize=(7,5))
rfm['Cluster'].value_counts().sort_index().plot(kind='bar', color='teal')
plt.title('Number of Customers per Cluster')
plt.xlabel('Cluster')
plt.ylabel('Number of Customers')
plt.tight_layout()
plt.show()

In [ ]:
print("=== MARKETING RECOMMENDATIONS ===")
print("""
Cluster 0 - High Value Customers:
- Reward with loyalty programmes
- Offer exclusive deals and early access

Cluster 1 - At Risk Customers:
- Send win-back email campaigns
- Offer special discounts to re-engage

Cluster 2 - New Customers:
- Send welcome offers
- Introduce loyalty programme benefits

Cluster 3 - Low Value Customers:
- Send low-cost promotions
- Focus budget on higher value segments
""")

In [ ]:
rfm.to_csv('../CustomerSegmentation_Results.csv', index=False)
print("Customer segmentation results saved successfully!")
print("Total customers segmented:", len(rfm))